# Pydantic

There are three main value props of Pydantic -
  * Validation: If the object ends up getting instantiated, it will have valid data. They make a point about not validating input data, but ensuring that the resulting object will always be valid. I think it refers to the data conversion that they do along the way. Data conversion is "deep" as seen in the `tastes` example below.
  * Serialization/deserialization
  * JSON schema generation



## Basics

Very similar to dataclasses, except all data model classes have to be derived from `BaseModel`. The class can be instantiated with keyword-only args.

In [1]:
%reset -f
from pydantic import * # type: ignore
from pprint import pprint
from datetime import datetime

In [2]:
class Cookie(BaseModel):
    flavor: str
    calories: int

In [3]:
chocolate_chip = Cookie(flavor="Chocoloate Chip", calories=200)

snicker_doodle_data = {"flavor": "Snicker Doodle", "calories": 220}
snicker_doodle = Cookie(**snicker_doodle_data)

In [4]:
chocolate_chip

Cookie(flavor='Chocoloate Chip', calories=200)

In [5]:
snicker_doodle.model_dump()

{'flavor': 'Snicker Doodle', 'calories': 220}

In [6]:
snicker_doodle.model_dump_json()

'{"flavor":"Snicker Doodle","calories":220}'

In [7]:
snicker_doodle.model_json_schema()

{'properties': {'flavor': {'title': 'Flavor', 'type': 'string'},
  'calories': {'title': 'Calories', 'type': 'integer'}},
 'required': ['flavor', 'calories'],
 'title': 'Cookie',
 'type': 'object'}

In [8]:
# supports both shallow and deep copy (if the model is nested)
# just pass deep=True flag to model_copy
cookie = snicker_doodle.model_copy()
id(cookie) == id(snicker_doodle)

False

The default behavior of data models is -
  * Mutable
  * Loss-less data conversion
  * No validation after instantiation
  * Extra attributes are silently swallowed

All these behaviors can be changed with model configs that are discussed below.  

In [9]:
# Mutable
chocolate_chip.flavor = "Double Chocolate Chip"
chocolate_chip

Cookie(flavor='Double Chocolate Chip', calories=200)

In [10]:
# Loss-less data conversion
# PyLance will rightly complain
Cookie(flavor="Oatmeal Raisin", calories="180")  # type: ignore

Cookie(flavor='Oatmeal Raisin', calories=180)

In [11]:
# No validation is done once the data object already exists
# PyLance is of course still doing its thing and flagging the below statement as a type error.
chocolate_chip.calories = "high"  # type: ignore
chocolate_chip

Cookie(flavor='Double Chocolate Chip', calories='high')

In [12]:
# Extra attributes are silently swallowed
# PyLance will rightly complain
Cookie(flavor="Oatmeal Raisin", calories=180, size="extra larage")  # type: ignore

Cookie(flavor='Oatmeal Raisin', calories=180)

### About Unions

If a field is a union of types, then the validation mechanism will choose the best type for the input called the "smart mode". There ways to customize this but for the most part, this default will suffice. In the below example, even though `"12"` would've been a valid int with data conversion, it is still taken in as a string because string is a better type match. 

In [13]:
class User(BaseModel):
    id: int | str

In [14]:
User(id="12")

User(id='12')

### About Dates
Looks like it does not support any arbitrary date format. It only supports some well known formats? If so, it'll be good to find out which ones.

In [15]:
class Holiday(BaseModel):
    name: str
    date: datetime

In [16]:
Holiday(name="Thanksgiving", date=datetime(2025, 11, 27))

Holiday(name='Thanksgiving', date=datetime.datetime(2025, 11, 27, 0, 0))

In [17]:
# Loss-less data conversion with PyLance complaining about it.
Holiday(name="Thanksgiving", date="2025-11-27")  # type: ignore

Holiday(name='Thanksgiving', date=datetime.datetime(2025, 11, 27, 0, 0))

In [19]:
try:
    Holiday(name="Thanksgiving", date="Thursday, November 27, 2025")  # type: ignore
except ValidationError as err:
    pprint(err.errors())

[{'ctx': {'error': 'invalid character in year'},
  'input': 'Thursday, November 27, 2025',
  'loc': ('date',),
  'msg': 'Input should be a valid datetime or date, invalid character in year',
  'type': 'datetime_from_date_parsing',
  'url': 'https://errors.pydantic.dev/2.11/v/datetime_from_date_parsing'}]


Even though JSON does not have support for dates, Pydantic does the neccessary magic to make it happen.

In [20]:
Holiday(name="Thanksgiving", date=datetime(2025, 11, 27)).model_dump_json()

'{"name":"Thanksgiving","date":"2025-11-27T00:00:00"}'

In [21]:
json_data = """
{
    "name": "Thanksgiviing",
    "date": "2025-11-27"
}
"""
Holiday.model_validate_json(json_data)

Holiday(name='Thanksgiviing', date=datetime.datetime(2025, 11, 27, 0, 0))

In [22]:
# More lossless data conversion with PyLance complaining
ts = datetime(2025, 11, 27).timestamp()
Holiday(name="Thanksgiving", date=ts)  # type: ignore

Holiday(name='Thanksgiving', date=datetime.datetime(2025, 11, 27, 8, 0, tzinfo=TzInfo(UTC)))

## Validating Data

Apart from the constructor, models can be validated using three other static methods, all of which return a fully instantiated data model object if possible -
  * `model_validate()`
  * `model_validate_json()`
  * `model_validate_strings()`

In [ ]:
%reset -f
from datetime import datetime
from pydantic import *  # type: ignore

In [24]:
class Cookie(BaseModel):
    flavor: str
    calories: int
    baked_on: datetime = datetime.now()

In [25]:
# Just like the constructor, but takes in a dict instead of **kwargs
Cookie.model_validate({"flavor": "Chocolate Chip", "calories": 200})

Cookie(flavor='Chocolate Chip', calories=200, baked_on=datetime.datetime(2025, 4, 6, 2, 13, 3, 472597))

In [26]:
# Takes in a JSON string.
cookie_data = """
{
    "flavor": "Chocolate Chip", 
    "calories": 200,
    "baked_on": "2025-04-03"
}
"""
Cookie.model_validate_json(cookie_data)

Cookie(flavor='Chocolate Chip', calories=200, baked_on=datetime.datetime(2025, 4, 3, 0, 0))

In [27]:
# Takes in a dict with string keys and string values
cookie_str = {"flavor": "Chocolate Chip", "calories": "200", "baked_on": "2025-04-03"}
Cookie.model_validate_strings(cookie_str)

Cookie(flavor='Chocolate Chip', calories=200, baked_on=datetime.datetime(2025, 4, 3, 0, 0))

I can also instantiate models without doing any validation. The documentation mentions three use cases for this -
  * performance reasons: when working with complex data as long as it is known to be valid.
  * when validator functions (covered later) are non-idempotent.
  * impure validtor functions that have side-effects I don't want to trigger.

In [28]:
Cookie.model_construct(
    flavor="Chocolate Chip", calories="very high", baked_on="last year"
)

Cookie(flavor='Chocolate Chip', calories='very high', baked_on='last year')

## Serialization

The main ways of serialization is using the [`model_dump()`](https://docs.pydantic.dev/latest/api/base_model/#pydantic.BaseModel.model_dump) and [`model_dump_json()`](https://docs.pydantic.dev/latest/api/base_model/#pydantic.BaseModel.model_dump_json) methods which support a bunch of different customization parameters.

In [30]:
%reset -f
from __future__ import annotations
from datetime import datetime
from pydantic import *  # type: ignore
from pprint import pprint
from enum import StrEnum, auto

In [31]:
class User(BaseModel):
    id: int
    created_at: datetime
    bookmarks: list[Bookmark] = []


class Bookmark(BaseModel):
    id: int
    added_at: datetime
    url: str
    content_type: ContentType


class ContentType(StrEnum):
    TEXT = auto()
    VIDEO = auto()
    PODCAST = auto()

In [32]:
user = User(
    id=1,
    created_at=datetime(2024, 5, 10, 18, 23),
    bookmarks=[
        Bookmark(
            id=100,
            added_at=datetime(2024, 5, 10, 19, 0),
            url="https://avilay.github.io/2020/12/12/should-i-just-watch-the-sunset.html",
            content_type=ContentType.TEXT,
        ),
        Bookmark(
            id=200,
            added_at=datetime(2024, 6, 10, 10, 20),
            url="https://youtube.com/playlist?list=PLWv7dKcFUITuj5o29j3Ak2CHG4nlEH-0f&si=wfokyE58E_X0yxyO",
            content_type=ContentType.VIDEO,
        ),
    ],
)
print(user.model_dump_json(indent=2))

{
  "id": 1,
  "created_at": "2024-05-10T18:23:00",
  "bookmarks": [
    {
      "id": 100,
      "added_at": "2024-05-10T19:00:00",
      "url": "https://avilay.github.io/2020/12/12/should-i-just-watch-the-sunset.html",
      "content_type": "text"
    },
    {
      "id": 200,
      "added_at": "2024-06-10T10:20:00",
      "url": "https://youtube.com/playlist?list=PLWv7dKcFUITuj5o29j3Ak2CHG4nlEH-0f&si=wfokyE58E_X0yxyO",
      "content_type": "video"
    }
  ]
}


A useful feature is the ability to add custom serialization. This is useful if I want to output the datetime field or an enum field in some specific format. I can set it at the field level or the model level using decorated methods defined in the model itself. Pydantic also supports a way to modify the output of the serialiation using `PlainSerializer` and `WrapSerializer` but I don't think I'll need those any time soon. The custom serialization can also use context data that is outside of the model to serialize. The context data is provided as part of the model dumpers.

In [35]:
%reset -f
from __future__ import annotations
from datetime import datetime
from pydantic import *  # type: ignore
from enum import StrEnum, auto
import pickle
from typing import Any

In [36]:
class User(BaseModel):
    id: int
    created_at: datetime
    bookmarks: list[Bookmark] = []

    @field_serializer("created_at")
    def serialize_created_at(self, created_at: datetime, _info) -> str:
        fmt = "%a, %b %d, %Y | %I:%M %p"
        return created_at.strftime(fmt)


class Bookmark(BaseModel):
    id: int
    added_at: datetime
    url: str
    content_type: ContentType

    @model_serializer
    def ser(self) -> dict[str, Any]:
        fmt = "%a, %b %d, %Y | %I:%M %p"
        return {
            "id": self.id,
            "added_at": self.added_at.strftime(fmt),
            "url": self.url,
            "content_type": self.content_type.upper(),
        }


class ContentType(StrEnum):
    TEXT = auto()
    VIDEO = auto()
    PODCAST = auto()

In [37]:
user = User(
    id=1,
    created_at=datetime(2024, 5, 10, 18, 23),
    bookmarks=[
        Bookmark(
            id=100,
            added_at=datetime(2024, 5, 10, 19, 0),
            url="https://avilay.github.io/2020/12/12/should-i-just-watch-the-sunset.html",
            content_type=ContentType.TEXT,
        ),
        Bookmark(
            id=200,
            added_at=datetime(2024, 6, 10, 10, 20),
            url="https://youtube.com/playlist?list=PLWv7dKcFUITuj5o29j3Ak2CHG4nlEH-0f&si=wfokyE58E_X0yxyO",
            content_type=ContentType.VIDEO,
        ),
    ],
)
print(user.model_dump_json(indent=2))

{
  "id": 1,
  "created_at": "Fri, May 10, 2024 | 06:23 PM",
  "bookmarks": [
    {
      "id": 100,
      "added_at": "Fri, May 10, 2024 | 07:00 PM",
      "url": "https://avilay.github.io/2020/12/12/should-i-just-watch-the-sunset.html",
      "content_type": "TEXT"
    },
    {
      "id": 200,
      "added_at": "Mon, Jun 10, 2024 | 10:20 AM",
      "url": "https://youtube.com/playlist?list=PLWv7dKcFUITuj5o29j3Ak2CHG4nlEH-0f&si=wfokyE58E_X0yxyO",
      "content_type": "VIDEO"
    }
  ]
}


A nice parameter to the various model dumpers is the serialization context. This is additional data that is outside of the model that I can use when serializing the model. The documentation has a good example that I am copying here.

In [38]:
class Content(BaseModel):
    text: str

    @field_serializer("text")
    def remove_stopwords(self, text: str, info: SerializationInfo) -> str:
        # This is whatever the user has supplied as part of the context param to the model dumper
        # In this case I know it will be a dict[str, list[str]]
        ctx = info.context
        if ctx:
            stopwords = ctx.get("stopwords", set())
            text = " ".join(
                word for word in text.split() if word.lower() not in stopwords
            )
        return text


content = Content(text="This is an example document")

In [39]:
# no stopwords
content.model_dump()

{'text': 'This is an example document'}

In [40]:
# with stopwords
content.model_dump(context={"stopwords": ["this", "is", "an"]})

{'text': 'example document'}

Pickling works as before without having to do anything special.

In [41]:
data = pickle.dumps(user)
data[:20]

b'\x80\x04\x95D\x02\x00\x00\x00\x00\x00\x00\x8c\x08__main_'

In [42]:
user2 = pickle.loads(data)
user2

User(id=1, created_at=datetime.datetime(2024, 5, 10, 18, 23), bookmarks=[Bookmark(id=100, added_at=datetime.datetime(2024, 5, 10, 19, 0), url='https://avilay.github.io/2020/12/12/should-i-just-watch-the-sunset.html', content_type=<ContentType.TEXT: 'text'>), Bookmark(id=200, added_at=datetime.datetime(2024, 6, 10, 10, 20), url='https://youtube.com/playlist?list=PLWv7dKcFUITuj5o29j3Ak2CHG4nlEH-0f&si=wfokyE58E_X0yxyO', content_type=<ContentType.VIDEO: 'video'>)])

## Model Config

Model config is the main way to configure type-level behavior of the model. These are controlled by the [ConfigDict](https://docs.pydantic.dev/latest/api/config/) which has a whole bunch of ways to customize the model behavior to be different from the default. During model definition I can set the `model_config` attribute. When using the validator static methods, I can pass in a `config=ConfigDict(...)` argument.

In [43]:
%reset -f

from pydantic import *  # type: ignore
from pprint import pprint

In [44]:
class User(BaseModel):
    id: int
    name: str = "APTG"

    model_config = ConfigDict(str_max_length=10)

In [45]:
user = User(id=1)
user

User(id=1, name='APTG')

In [46]:
try:
    user = User(id=1, name="x" * 20)
except ValidationError as err:
    pprint(err.errors())

[{'ctx': {'max_length': 10},
  'input': 'xxxxxxxxxxxxxxxxxxxx',
  'loc': ('name',),
  'msg': 'String should have at most 10 characters',
  'type': 'string_too_long',
  'url': 'https://errors.pydantic.dev/2.11/v/string_too_long'}]


There are more ways to configure the model when using Python's builtin `dataclass` or `TypedDict`. But my current thinking is that I should not mix `dataclass` and `BaseModel`, it might make debugging down the line harder. And using `TypedDict` is just a bad code smell.

I can use model config on type adapters as well. But the example I found in the documentation was pretty lame. And I have not been able to come up with a better example by myself so far.

In [47]:
Strings = TypeAdapter(list[str], config=ConfigDict(coerce_numbers_to_str=True))
vals = Strings.validate_python([1, "2", 3.14])
vals

['1', '2', '3.14']

If I am building a heirarchy of classes, then I can specify the model config in root model and every other model inherits that config.

In [48]:
class Parent(BaseModel):
    model_config = ConfigDict(str_to_upper=True)


class Model(Parent):
    x: str


Model(x="foo").model_dump_json()

'{"x":"FOO"}'

## Frozen

Make the model pseudo-immutable. There are two ways of doing this - one using the model config and another using an attribute of the `BaseModel`. I prefer using the model config because it keeps my programming model consistent.

In [49]:
%reset -f
from pydantic import *  # type: ignore
from pprint import pprint

In [50]:
class Cookie(BaseModel):
    flavor: str
    calories: int

    model_config = ConfigDict(frozen=True)

In [51]:
chocolate_chip = Cookie(flavor="Chocolate Chip", calories=200)
try:
    chocolate_chip.calories = 220
except ValidationError as err:
    pprint(err.errors())

[{'input': 220,
  'loc': ('calories',),
  'msg': 'Instance is frozen',
  'type': 'frozen_instance',
  'url': 'https://errors.pydantic.dev/2.11/v/frozen_instance'}]


## Extra Fields
By default, any extra fields specified on the input are swallowed siliently, no errors, but no sign of the data either. I can change this behavior with the `extra` model config which can have three values - `ignore` which is the default, `allow` which retains the extra data, and `forbid` which errors out.

In [52]:
%reset -f
from pydantic import *  # type: ignore
from pprint import pprint

In [53]:
class User(BaseModel):
    id: int
    name: str

In [54]:
# By default, any extra fields are swallowed silently, no errors, but no sign of that data either.
User(
    id=1,
    name="AP",
    hobbies=["Car Racing", "Video Games"],
)

User(id=1, name='AP')

In [55]:
# I can use model config to modify this behavior, s.t that the extra fields' data will be retained.
class ChillUser(BaseModel):
    id: int
    name: str = "APTG"

    model_config = ConfigDict(str_max_length=10, extra="allow")


ChillUser(id=1, hobbies=["Car Racing", "Video Games"])

ChillUser(id=1, name='APTG', hobbies=['Car Racing', 'Video Games'])

In [56]:
class NoChillUser(BaseModel):
    id: int
    name: str = "APTG"

    model_config = ConfigDict(extra="forbid")


try:
    NoChillUser(id=1, hobbies=["Car Racing", "Video Games"])
except ValidationError as err:
    pprint(err.errors())

[{'input': ['Car Racing', 'Video Games'],
  'loc': ('hobbies',),
  'msg': 'Extra inputs are not permitted',
  'type': 'extra_forbidden',
  'url': 'https://errors.pydantic.dev/2.11/v/extra_forbidden'}]


## Data Conversion

The default behavior is to convert an input of a general type to a model attribute of a more specific type as long as that conversion can happen without loss of data. This [conversion table](https://docs.pydantic.dev/latest/concepts/conversion_table/) lists how various types are converted to what.

In [57]:
%reset -f
from pydantic import *  # type: ignore
from pprint import pprint

In [58]:
class IntValue(BaseModel):
    value: int

In [59]:
# OK - no conversion required
IntValue(value=1)

IntValue(value=1)

In [60]:
# Even though the input is a float, it can be converted into an int without loss of data so - OK
IntValue(value=1.0)  # type: ignore

IntValue(value=1)

In [61]:
# This float cannot be converted to an int without losing data so - ERROR
try:
    i = IntValue(value=2.1)
    print(i)
except ValidationError as err:
    pprint(err.errors())

[{'input': 2.1,
  'loc': ('value',),
  'msg': 'Input should be a valid integer, got a number with a fractional part',
  'type': 'int_from_float',
  'url': 'https://errors.pydantic.dev/2.11/v/int_from_float'}]


In [62]:
# No loss of data - OK
# Python's int("1") will also work just like this
IntValue(value="1")  # type: ignore

IntValue(value=1)

In [63]:
# Data loss upon conversion - ERROR
try:
    IntValue(value="2.7")
except ValidationError as err:
    pprint(err.errors())

[{'input': '2.7',
  'loc': ('value',),
  'msg': 'Input should be a valid integer, unable to parse string as an '
         'integer',
  'type': 'int_parsing',
  'url': 'https://errors.pydantic.dev/2.11/v/int_parsing'}]


There is this weird bit of documentation that says that I should choose to make my field types as specific as possible, so use `list[int]` instead of `Sequence[int]`. 

### Strict Mode
To prevent automatic data conversion, I can specify the strict mode at three different places -
  * per-model
  * per-field
  * per-validation-call

In [64]:
class Cookie(BaseModel):
    flavor: str
    calories: int

    model_config = ConfigDict(strict=True)

In [65]:
try:
    Cookie(flavor="Chocolate Chip", calories="200")
except ValidationError as err:
    pprint(err.errors())

[{'input': '200',
  'loc': ('calories',),
  'msg': 'Input should be a valid integer',
  'type': 'int_type',
  'url': 'https://errors.pydantic.dev/2.11/v/int_type'}]


In [66]:
try:
    IntValue.model_validate({"value": 200.0}, strict=True)
except ValidationError as err:
    pprint(err.errors())

[{'input': 200.0,
  'loc': ('value',),
  'msg': 'Input should be a valid integer',
  'type': 'int_type',
  'url': 'https://errors.pydantic.dev/2.11/v/int_type'}]


In [67]:
class Rectangle(BaseModel):
    length: int
    breadth: int = Field(strict=True)

In [69]:
Rectangle(length=100.0, breadth=100)  # type: ignore

Rectangle(length=100, breadth=100)

In [70]:
try:
    Rectangle(length=100.0, breadth=100.0)
except ValidationError as err:
    pprint(err.errors())

[{'input': 100.0,
  'loc': ('breadth',),
  'msg': 'Input should be a valid integer',
  'type': 'int_type',
  'url': 'https://errors.pydantic.dev/2.11/v/int_type'}]


## Arbitrary Field Types
Fields in Pydantic models can only have "Pydantic Compatible" types. AFAIK - this means that they can only be primitive types, types defined in Python's stdlib, types wrapped in the special `InstanceOf` type, or other BaseModel derived types. There are ways to support fields with arbitrary types, but they come with a cost - either the validation suffers, or I have to write extra code to make everything work. It will not work OOB.

In [71]:
%reset -f
from pydantic import *  # type: ignore
from pprint import pprint

In [72]:
class A:
    def __init__(self, cookie: int) -> None:
        self.cookie = cookie


class B(BaseModel):
    cookie: int


# The following will work because B is a BaseModel itself.
class MyModelB(BaseModel):
    b: B


model_b = MyModelB(b=B(cookie=1))
pprint(model_b)

# The following will not work because A is not a BaseModel
try:

    class MyModelA(BaseModel):
        a: A

except PydanticSchemaGenerationError as err:
    print("\n", err.message)

MyModelB(b=B(cookie=1))

 Unable to generate pydantic-core schema for <class '__main__.A'>. Set `arbitrary_types_allowed=True` in the model_config to ignore this error or implement `__get_pydantic_core_schema__` on your type to fully support it.

If you got this error by calling handler(<some type>) within `__get_pydantic_core_schema__` then you likely need to call `handler.generate_schema(<some type>)` since we do not call `__get_pydantic_core_schema__` on `<some type>` otherwise to avoid infinite recursion.


If I want to make it work, I can either wrap `A` inside `IsntanceOf` or configure the model to allow arbitrary types. In either case, these fields won't be validated.

In [73]:
class MyModelA(BaseModel):
    a: InstanceOf[A]


MyModelA(a=A(1))
MyModelA(a=A("hahaha"))  # No validation

MyModelA(a=<__main__.A object at 0x70eef5deead0>)

In [74]:
class MyModelA(BaseModel):
    a: A
    model_config = ConfigDict(arbitrary_types_allowed=True)


MyModelA(a=A(1))

# PyLance will complain with a type error, but Pydantic will let this go through.
MyModelA(a=A("hahaha"))  # type: ignore

MyModelA(a=<__main__.A object at 0x70eef5da6fd0>)

In [75]:
# If the type is already defined in Python's stdlib, it is considereed as a "Pydantic Compatible" type I think!

from pathlib import Path


class FilePath(BaseModel):
    file: Path

In [76]:
FilePath(file=Path.home())

FilePath(file=PosixPath('/home/avilay'))

In [78]:
# The lossless data conversion works as well
# As usual PyLance will whine about it with a type error.
FilePath(file="/home/avilay/Documents")  # type: ignore

FilePath(file=PosixPath('/home/avilay/Documents'))

## Nested Models

Nested models, where a field's type is another base model, don't have model configs propagate to all the fields.

In [79]:
%reset -f
from pydantic import *  # type: ignore

In [80]:
class A(BaseModel):
    name: str


class B(BaseModel):
    a: A
    title: str

    # This will only apply to str fields of B, not "recursively" to A
    model_config = ConfigDict(str_max_length=4)


B(title="aaaa", a=A(name="A very long long name"))

B(a=A(name='A very long long name'), title='aaaa')

## Forward References

Supports the standard forward references scenarios - referencing a model that has not been defined yet, and recursive references. It is possible to define models with cyclic references. Pydantic will find this error. It does so without stackoverflow so I can do some fancy error handling if so wish.

In [81]:
%reset -f
from __future__ import annotations
from pydantic import *  # type: ignore
from pprint import pprint
from typing import Any

In [82]:
class B(BaseModel):
    a: A

In [83]:
class A(BaseModel):
    name: str

In [84]:
B(a=A(name="A"))

B(a=A(name='A'))

In [85]:
class Node(BaseModel):
    data: int
    next: Node | None = None

In [86]:
Node(data=10, next=Node(data=20))

Node(data=10, next=Node(data=20, next=None))

In [87]:
x: dict[str, Any] = {"data": 10}
y = {"data": 20}
x["next"] = y
y["next"] = x

try:
    Node.model_validate(x)
except ValidationError as err:
    pprint(err.errors())

[{'input': {'data': 10,
            'next': {'data': 20,
                     'next': <Recursion on dict with id=124171627978944>}},
  'loc': ('next', 'next'),
  'msg': 'Recursion error - cyclic reference detected',
  'type': 'recursion_loop',
  'url': 'https://errors.pydantic.dev/2.11/v/recursion_loop'}]


## Type Adapters and `list`s

According to the [documentation](https://docs.pydantic.dev/latest/concepts/type_adapter/) `TypeAdapters` work with non-base model types as well as lists of base model types, but I did not find any examples of the former. And when I tried to make it work with the example above, it didn't quite work.

For now I'll just jot down the usage for lists. The basic idea is that, if I have a list of base models, I can still validate and instantiate the right types without having to create another BaseModel just for the list.

In [88]:
class Cookie(BaseModel):
    flavor: str
    calories: int


Cookies = TypeAdapter(list[Cookie])
cookies_data = [
    {"flavor": "Chocolate Chip", "calories": 200},
    {"flavor": "Snicker Doodle", "calories": 220},
]
cookies = Cookies.validate_python(cookies_data)
cookies

[Cookie(flavor='Chocolate Chip', calories=200),
 Cookie(flavor='Snicker Doodle', calories=220)]

## Generic Models

Works pretty much like generic types in Python itself. Supports the new syntax and all. When using bounded types, make sure to specify the type parameter when instantiating the data object. If done incorrectly, it might lead to data loss as shown below.

In [89]:
%reset -f
from pydantic import *  # type: ignore

In [90]:
class Cookie(BaseModel):
    flavor: str
    calories: int


class Snack[T: Cookie](BaseModel):
    cookie: T
    serving_size: int


class LowFatCookie(Cookie):
    carbs_gms: int

In [91]:
# OK: Instantiating with the correct type parameter set to LowFatCookie.
Snack[LowFatCookie](
    cookie=LowFatCookie(flavor="Chocolate Chip", calories=100, carbs_gms=25),
    serving_size=2,
).model_dump()

{'cookie': {'flavor': 'Chocolate Chip', 'calories': 100, 'carbs_gms': 25},
 'serving_size': 2}

In [92]:
# OK: Even when the type parameter is left out, it will still work because there are on wonky default types.
Snack(
    cookie=LowFatCookie(flavor="Chocolate Chip", calories=100, carbs_gms=25),
    serving_size=2,
).model_dump()

{'cookie': {'flavor': 'Chocolate Chip', 'calories': 100, 'carbs_gms': 25},
 'serving_size': 2}

In [93]:
# Type param is Cookie even though the data ha LowFatCookie. carbs_gms is lost.
Snack[Cookie](
    cookie=LowFatCookie(flavor="Chocolate Chip", calories=100, carbs_gms=25),
    serving_size=2,
).model_dump()

{'cookie': {'flavor': 'Chocolate Chip', 'calories': 100}, 'serving_size': 2}

## Dynamic Model Creation

Creates the model class at runtime. Here is a simple example. More advanced examples in the [docs](https://docs.pydantic.dev/latest/concepts/models/#dynamic-model-creation).

In [94]:
%reset -f
from pydantic import *  # type: ignore

In [95]:
Cookie = create_model("Cookie", calories=int, flavor=(str, "Chocolate Chip"))
Cookie

__main__.Cookie

The above code is equivalent to -
```python
class Cookie(BaseModel):
    calories: int
    flaovr: str = "Chocolate Chip"
```

In [96]:
Cookie(calories=200)

Cookie(calories=200, flavor='Chocolate Chip')

## Root models
If I want to define a Pydantic data model made up of Python types without going through the hassle of authoring a class and attributes, etc. I can use root models.

In [102]:
%reset -f
from pydantic import *  # type: ignore
from pprint import pprint
from typing import Iterator

In [98]:
Hobbies = RootModel[list[str]]

In [99]:
hobbies = Hobbies(["Car Racing", "Video Games"])
hobbies

RootModel[list[str]](root=['Car Racing', 'Video Games'])

In [100]:
pprint(hobbies.model_dump())
pprint(hobbies.model_dump_json())

['Car Racing', 'Video Games']
'["Car Racing","Video Games"]'


In [101]:
# But I cannot index into it.
try:
    hobbies[0]
except TypeError as err:
    pprint(err)

TypeError("'RootModel[list[str]]' object is not subscriptable")


In [103]:
# If I want better semantics like indexing I need to implement my own __getitem__ and __iter__ methods
class StockTicker(RootModel):
    root: dict[str, float]

    def __getitem__(self, idx: str) -> float:
        return self.root[idx]

    def __iter__(self) -> Iterator[str]:
        yield from self.root.keys()

In [104]:
ticker = StockTicker({"APPL": 200, "MSFT": 240})

In [105]:
ticker["APPL"]

200.0

In [106]:
for stock in ticker:
    print(stock, ticker[stock])

APPL 200.0
MSFT 240.0


## Automatically Excluded Attributes

Model attributes marked with a `ClassVar` or having their name start with a single underscore are excluded from the schema. The private instance attributes are not even created during `__init__` and there is no way I can access it from either within or outside the model. The only way to use them is to first intialize them in the `model_post_init` method.

In [108]:
%reset -f
from pydantic import *  # type: ignore
from typing import ClassVar, Any

In [109]:
class User(BaseModel):
    connstr: ClassVar[str] = "sqlite://exampledb"
    # Without the model_post_init this is useless
    _partition_id: int
    id: int
    name: str

In [110]:
User.model_json_schema()

{'properties': {'id': {'title': 'Id', 'type': 'integer'},
  'name': {'title': 'Name', 'type': 'string'}},
 'required': ['id', 'name'],
 'title': 'User',
 'type': 'object'}

In [111]:
user = User(_partition_id=0, id=1, name="APTG")
user

User(id=1, name='APTG')

In [112]:
user.connstr

'sqlite://exampledb'

In [113]:
User.connstr

'sqlite://exampledb'

In [114]:
try:
    user._partition_id
except AttributeError as err:
    print(err)

'User' object has no attribute '_partition_id'


In [115]:
class Photo(BaseModel):
    connstr: ClassVar[str] = "sqlite://exampledb"
    _partition_id: int

    id: int
    title: str

    def model_post_init(self, context: Any) -> None:
        self._partition_id = self.id % 10

In [117]:
photo = Photo(id=1, title="Vacation")  # type: ignore

In [118]:
photo._partition_id

1

## Fields
The [`Field`](https://docs.pydantic.dev/latest/api/fields/) function is used to set constraints on a per field basis. The linked documentation has all the different constraints/rules I can add to a field.

In [119]:
%reset -f
from datetime import datetime
from pydantic import *  # type: ignore
from pprint import pprint

In [120]:
class CookieJar(BaseModel):
    cookies: list[str] = Field(min_length=1, max_length=10)
    capacity: int
    filled_on: datetime = datetime.now()

In [121]:
jar = CookieJar(cookies=["Snicker Doodle"], capacity=10)
jar

CookieJar(cookies=['Snicker Doodle'], capacity=10, filled_on=datetime.datetime(2025, 4, 6, 2, 25, 0, 596971))

The [PyDantic documentation](https://docs.pydantic.dev/latest/concepts/fields/#the-annotated-pattern) states that I should prefer using [Python Annotations](https://docs.python.org/3/library/typing.html#typing.Annotated) to add meta-data to my fields. One of the uses they cite is to build reusable types. But I think I'll stick to using `Field` function as not to mix Python typing esoterica with Pydantic. I can still make reusable types as demonstrated below.

In [122]:
SmallList = Field(min_length=1, max_length=3)


class PillBox(BaseModel):
    box: list[str] = SmallList
    capacity: int

In [123]:
PillBox(box=["A", "B"], capacity=10)

PillBox(box=['A', 'B'], capacity=10)

In [124]:
try:
    PillBox(box=["A", "B", "C", "D"], capacity=10)
except ValidationError as err:
    pprint(err.errors())

[{'ctx': {'actual_length': 4, 'field_type': 'List', 'max_length': 3},
  'input': ['A', 'B', 'C', 'D'],
  'loc': ('box',),
  'msg': 'List should have at most 3 items after validation, not 4',
  'type': 'too_long',
  'url': 'https://errors.pydantic.dev/2.11/v/too_long'}]


## Default Values
As far as I can tell, there are three ways to set default values -
  * With the `field_name: type = default_val` declaration
  * Specifying in the `Field` function `field_name: type = Field(default=default_val)`
  * Specifying a default factory in the `Field` function `field_name: type = Field(default_factory: func)`

Just like in normal Python, don't use mutable objects like empty lists or dicts as defaults. Even though Pydantic will automatically do the right thing by making copies and what not, it is just a bad code smell.  

In [125]:
%reset -f
from pydantic import *  # type: ignore
from pprint import pprint
import random
from uuid import uuid4

In [126]:
class Cookie(BaseModel):
    flavor: str = Field(default="Chocolate Chip")
    calories: int = Field(default_factory=lambda: random.randint(100, 200))

In [127]:
Cookie()

Cookie(flavor='Chocolate Chip', calories=187)

The default factory can also take a single required argument, in which case the already validated data will be passed as a dictionary.

In [128]:
class User(BaseModel):
    id: str = Field(default_factory=lambda: uuid4().hex)
    email: str | None = None
    name: str = Field(
        default_factory=lambda data: data["email"] if data["email"] else "Anon"
    )

In [129]:
User()

User(id='08e47a70ee784b62882d2978bb61acb0', email=None, name='Anon')

In [130]:
User(email="cookie@monster.com")

User(id='004c702bc99f443a963266e8065acbee', email='cookie@monster.com', name='cookie@monster.com')

In [131]:
User(email="cookie@monster.com", name="Cookie Monster")

User(id='b579069cfdd140b6bf37a3c679d9747d', email='cookie@monster.com', name='Cookie Monster')

The order of declaration of fields is important for this to work. If `name` had been defined before `email`, the input `data` would not have included `email`!

By default, default values are not validated. I can change this behavior either with a field argument or a with a model config.

In [132]:
class Cookie(BaseModel):
    calories: int = "hello"
    flavor: str

    model_config = ConfigDict(validate_default=True)

In [133]:
try:
    Cookie(flavor="Chocolate Chip")
except ValidationError as err:
    pprint(err.errors())

[{'input': 'hello',
  'loc': ('calories',),
  'msg': 'Input should be a valid integer, unable to parse string as an '
         'integer',
  'type': 'int_parsing',
  'url': 'https://errors.pydantic.dev/2.11/v/int_parsing'}]


In [134]:
class Rectangle(BaseModel):
    length: float
    breadth: int = Field(
        default_factory=lambda data: data["length"], validate_default=True
    )

In [135]:
Rectangle(length=10)

Rectangle(length=10.0, breadth=10)

In [136]:
try:
    Rectangle(length=10.1)
except ValidationError as err:
    pprint(err.errors())

[{'input': 10.1,
  'loc': ('breadth',),
  'msg': 'Input should be a valid integer, got a number with a fractional part',
  'type': 'int_from_float',
  'url': 'https://errors.pydantic.dev/2.11/v/int_from_float'}]


## Alias
My general perception is that the alias implementation is very wonky and unintuitive. Its best to avoid using it if possbile. The only use case that I can think of is when interacting with different systems that might have different naming conventions - specifically `camelCase` and `snake_case`. I have a recipe below for that.

In [137]:
%reset -f
from pydantic import *  # type: ignore
from pprint import pprint

In [138]:
from pydantic.alias_generators import to_camel


class Person(BaseModel):
    first_name: str
    last_name: str

    model_config = ConfigDict(
        alias_generator=to_camel, validate_by_alias=True, validate_by_name=True
    )

In [139]:
# Initialization works with snake case
Person(first_name="Avilay", last_name="Parekh")

Person(first_name='Avilay', last_name='Parekh')

In [140]:
# Initialization also works with camel case
ap = Person(firstName="Avilay", lastName="Parekh")  # type: ignore
ap

Person(first_name='Avilay', last_name='Parekh')

In [141]:
# Serialization with camel case needs the by_alias flag
ap.model_dump(by_alias=True)

{'firstName': 'Avilay', 'lastName': 'Parekh'}

In [142]:
# Default serialization is with snake case
ap.model_dump()

{'first_name': 'Avilay', 'last_name': 'Parekh'}

In [143]:
# Field access is only through snake case
ap.first_name = "APTG"

In [144]:
# Field access with camel case does not work
try:
    ap.lastName = "hahaha"
except ValueError as err:
    print(err)

"Person" object has no field "lastName"


## Constraints
I can add a bunch of constraints to individual fields that will be followed during validation.

  * Numeric Constraints
    - `gt`: greater than
    - `lt`: less than
    - `ge`: greater than equal to
    - `le`: less than equal to
    - `multipole_of`: a multiple of a given number
    - `allow_inf_nan`: allow infinity and nan floats
  * Iterable/String Constraints
    - `min_length`: Minimum length of the string
    - `max_length`: Maximum length of the string
    - `pattern`: A regex pattern that the input must match (obviously only applies to string)
  * Decimal Constraints
    - `max_digits`: Maximum number of digits with `Decimal` object.
    - `decimal_places`: Maximum number of decimal places allowed.

In [145]:
class FunWithNumbers(BaseModel):
    strictly_positive: int = Field(gt=0)
    positive: float = Field(ge=0)
    logscale: int = Field(multiple_of=10)
    anything: float = Field(allow_inf_nan=True)

In [146]:
FunWithNumbers(strictly_positive=1, positive=0, logscale=100, anything=float("inf"))

FunWithNumbers(strictly_positive=1, positive=0.0, logscale=100, anything=inf)

In [147]:
try:
    FunWithNumbers(strictly_positive=0, positive=-1, logscale=100, anything=-2.3)
except ValidationError as err:
    pprint(err.errors())

[{'ctx': {'gt': 0},
  'input': 0,
  'loc': ('strictly_positive',),
  'msg': 'Input should be greater than 0',
  'type': 'greater_than',
  'url': 'https://errors.pydantic.dev/2.11/v/greater_than'},
 {'ctx': {'ge': 0.0},
  'input': -1,
  'loc': ('positive',),
  'msg': 'Input should be greater than or equal to 0',
  'type': 'greater_than_equal',
  'url': 'https://errors.pydantic.dev/2.11/v/greater_than_equal'}]


## Validation
A lot of validation can be controlled by using the right type for a field. In case the validation logic cannot be represented with a type, we can write our own validation function. There are the following ways to do validation in Pydantic:

  * Using [custom types](https://docs.pydantic.dev/latest/api/types/#pydantic.types) that Pydantic has defined.
  * Specifying constraints on fields.
  * Custom validation functions.

In [149]:
%reset -f
from pydantic import *  # type: ignore
from pprint import pprint
from datetime import datetime
import cprint  # type: ignore
from typing import Self

### Custom Types
These are pretty straightforward to use. Just annotate the type with this custom type. Pydantic has a long list of interesting custom types that should meet most of my requirements!

In [150]:
class User(BaseModel):
    id: int
    created_at: PastDatetime
    password: SecretStr

In [152]:
user = User(id=1, created_at=datetime(2024, 1, 1), password="passworld")  # type: ignore
user.model_dump()

{'id': 1,
 'created_at': datetime.datetime(2024, 1, 1, 0, 0),
 'password': SecretStr('**********')}

In [153]:
user.model_dump_json()

'{"id":1,"created_at":"2024-01-01T00:00:00","password":"**********"}'

## Validation Functions
These can be at a per-field level or at the model level. In either case there are four different types of custom validators depending on when in the validation lifecycle they run -
  * Before validators - run before the Pydantic validators run.
  * After validators - run after the Pydantic validators run.
  * Plain validators - run instead of Pydantic validators, i.e., the Pydantic validators never run.
  * Wrap validators - wrap Pydantic validators, and they can choose if and when to call the Pydantic validators.

The per-field validators are run as each field is processed, but before the full object is instantiated, this means that these functions need to be class methods. The model validators on the other hand can be instance methods if the validator is supposed to be run after. In all other cases they still have to be class methods.  

If the custom function wants to raise an error, the simplest thing to do is to raise the plain old `ValueError`. Pydantic will convert this into the usual `ValidationError`. However, I do have the option to raise a couple of other errors like `AssertionError` or `PydanticCustomError` for advanced scenarios.

The validator functions can also optionally get additional validation data and validation context, but this too seems like advanced scenarios that I don't want to get into unless I actually need to.

Pydnantic also has a cool feature where I can decorate any old Python function with type annotations with `@validate_call` and get the same benefits of a model valiator. For now, I am not going to use while I get more familiar with Python's own type annotations.

### Before Validators

Before-validators are given the user input directly, which means the input can be anything. The validator needs to be robust to any and all sorts of input. As can be seen in the model before-validator, it has to deal with a bunch of different edge cases.

In [154]:
USERNAMES = ["cookie", "happy", "orange"]


class User(BaseModel):
    id: int
    name: str

    # The input can be Any-thing
    @field_validator("name", mode="before")
    def is_unique(cls, value: Any) -> str:
        cprint.cyan_print(f"type={type(value)}, {value}")
        if isinstance(value, str):
            if value not in USERNAMES:
                return value
            else:
                raise ValueError("User name already exists!")
        else:
            raise ValueError("User name must be a string!")

In [155]:
User(id=1, name="APTG")

type=<class 'str'>, APTG


User(id=1, name='APTG')

In [156]:
try:
    User(id=2, name="cookie")
except ValidationError as err:
    pprint(err.errors())

type=<class 'str'>, cookie
[{'ctx': {'error': ValueError('User name already exists!')},
  'input': 'cookie',
  'loc': ('name',),
  'msg': 'Value error, User name already exists!',
  'type': 'value_error',
  'url': 'https://errors.pydantic.dev/2.11/v/value_error'}]


In [157]:
try:
    User(id=3, name={"username": "Anon"})
except ValidationError as err:
    pprint(err.errors())

type=<class 'dict'>, {'username': 'Anon'}
[{'ctx': {'error': ValueError('User name must be a string!')},
  'input': {'username': 'Anon'},
  'loc': ('name',),
  'msg': 'Value error, User name must be a string!',
  'type': 'value_error',
  'url': 'https://errors.pydantic.dev/2.11/v/value_error'}]


In [159]:
%reset -f
from __future__ import annotations
from pydantic import *  # type: ignore
from enum import StrEnum, auto
from datetime import datetime
import cprint  # type: ignore

In [160]:
def reformat_date(datestr: str) -> str:
    fmts = [
        # Mon, Dec 01, 1980 | 03:33 PM
        "%a, %b %d, %Y | %I:%M %p",
        # 12-01-80 15:33
        "%m-%d-%y %H:%M",
    ]
    for fmt in fmts:
        try:
            ds = datetime.strptime(datestr, fmt)
            return ds.isoformat()
        except ValueError:
            pass
    return datestr


class User(BaseModel):
    id: int
    created_at: datetime
    bookmarks: list[Bookmark] = []

    @model_validator(mode="before")
    @classmethod
    def sanitize(cls, data: Any) -> Any:
        cprint.cyan_print(f"type={type(data)}, {data}")
        if not isinstance(data, dict):
            return data

        # Reformat all the dates in the model hierarchy
        if "created_at" in data and isinstance(data["created_at"], str):
            data["created_at"] = reformat_date(data["created_at"])

        for bookmark in data.get("bookmarks", []):
            if "added_at" in bookmark and isinstance(bookmark["added_at"], str):
                bookmark["added_at"] = reformat_date(bookmark["added_at"])

            # StrEnum likes its value in lower case
            if "content_type" in bookmark and isinstance(bookmark["content_type"], str):
                bookmark["content_type"] = bookmark["content_type"].lower()
        return data


class Bookmark(BaseModel):
    id: int
    added_at: datetime
    url: str
    content_type: ContentType


class ContentType(StrEnum):
    TEXT = auto()
    VIDEO = auto()
    PODCAST = auto()

In [161]:
user_data = """
{
  "id": 1,
  "created_at": "Fri, May 10, 2024 | 06:23 PM",
  "bookmarks": [
    {
      "id": 100,
      "added_at": "Fri, May 10, 2024 | 07:00 PM",
      "url": "https://avilay.github.io/2020/12/12/should-i-just-watch-the-sunset.html",
      "content_type": "TEXT"
    },
    {
      "id": 200,
      "added_at": "Mon, Jun 10, 2024 | 10:20 AM",
      "url": "https://youtube.com/playlist?list=PLWv7dKcFUITuj5o29j3Ak2CHG4nlEH-0f&si=wfokyE58E_X0yxyO",
      "content_type": "VIDEO"
    }
  ]
}
"""
User.model_validate_json(user_data)

type=<class 'dict'>, {'id': 1, 'created_at': 'Fri, May 10, 2024 | 06:23 PM', 'bookmarks': [{'id': 100, 'added_at': 'Fri, May 10, 2024 | 07:00 PM', 'url': 'https://avilay.github.io/2020/12/12/should-i-just-watch-the-sunset.html', 'content_type': 'TEXT'}, {'id': 200, 'added_at': 'Mon, Jun 10, 2024 | 10:20 AM', 'url': 'https://youtube.com/playlist?list=PLWv7dKcFUITuj5o29j3Ak2CHG4nlEH-0f&si=wfokyE58E_X0yxyO', 'content_type': 'VIDEO'}]}


User(id=1, created_at=datetime.datetime(2024, 5, 10, 18, 23), bookmarks=[Bookmark(id=100, added_at=datetime.datetime(2024, 5, 10, 19, 0), url='https://avilay.github.io/2020/12/12/should-i-just-watch-the-sunset.html', content_type=<ContentType.TEXT: 'text'>), Bookmark(id=200, added_at=datetime.datetime(2024, 6, 10, 10, 20), url='https://youtube.com/playlist?list=PLWv7dKcFUITuj5o29j3Ak2CHG4nlEH-0f&si=wfokyE58E_X0yxyO', content_type=<ContentType.VIDEO: 'video'>)])

### After Validators
After validators are guaranteed to get the actual declared type of the field. If it is a model validator it can be an instance method as opposed to class methods for every other type of validator. For instance methods, the return type is `Self` which is a shorthand for a generic self.

```python
class MyClass:
    def func[T: MyClass](self: T) -> T:
        ....
        return self
```

Can be re-written as -
```python
class MyClass:
    def funct(self) -> Self:
        .....
        return self
```

In [162]:
class Counter(BaseModel):
    value: int

    @model_validator(mode="after")
    def power_of_two(self) -> Self:
        val = self.value
        val = val if val > 0 and (val & (val - 1)) == 0 else 1
        self.value = val
        return self

In [163]:
Counter(value=2)

Counter(value=2)

In [164]:
Counter(value="008")  # type: ignore

Counter(value=8)

### Wrap Validators

These functions get a handler as input that can be called to run the Pydantic validation. The function can choose if/when to run it.

In [165]:
class Cookie(BaseModel):
    flavor: str
    calories: PositiveInt


class CookieJar(BaseModel):
    cookies: list[Cookie] = Field(max_length=3)
    capacity: PositiveInt

    @field_validator("cookies", mode="wrap")
    @classmethod
    def clip_cookies(
        cls, cookies: Any, handler: ValidatorFunctionWrapHandler
    ) -> list[Cookie]:
        # Do the usual type checks
        if len(cookies) > 3:
            cookies = cookies[:3]
        return handler(cookies)

In [166]:
CookieJar(
    capacity=10,
    cookies=[
        Cookie(flavor="Chocolate Chip", calories=200),
        Cookie(flavor="Snicker Doodle", calories=220),
        Cookie(flavor="Oatmeal Raisin", calories=180),
        Cookie(
            flavor="Double Chocolate Chip", calories=240
        ),  # this will be clipped off
    ],
)

CookieJar(cookies=[Cookie(flavor='Chocolate Chip', calories=200), Cookie(flavor='Snicker Doodle', calories=220), Cookie(flavor='Oatmeal Raisin', calories=180)], capacity=10)

In [167]:
class Fruit:
    def __repr__(self) -> str:
        return self.__class__.__name__


class Banana(Fruit): ...


class Apple(Fruit): ...


class Basket(BaseModel):
    fruits: list[InstanceOf[Fruit]]

---

# Appendix

## Misc Fields Support
There are a bunch of other things I can do with the `Field` function like controlling how or whether it will be outputted, marking them for deprecation, etc. 

## Alias Details
Some more details on alias in case I have to use it.

I can have different names for my fields. Here are some use cases I can think for these -
  * Accomodate different naming conventions across different systems
  * Seperate out the internal data model from the application model (?)

The default "alias" is implemented in a weird way. When I am instantiating the model I have to use the alias, but when accessing that field on the data object, I have to use the "real" name of the field. When the model is serialized, again the real name is used.

I think in order to compensate for this design choice, Pydantic has introduced the concept of seralization and validation aliases. Serialization alias is to be used for accessing the fields on the data object and when the model is serialized. Validation alias is to be used when instantiating the model.

In [ ]:
class User(BaseModel):
    name: str = Field(alias="username")

In [ ]:
# Use the alias at initialization
user = User(username="APTG")
user

In [ ]:
# Serialization will use the real name
user.model_dump()

In [ ]:
# Field access is also via the real name
user.name

In [ ]:
# Trying to access the alias will throw a plain old Python error
try:
    user.username
except AttributeError as err:
    print(err)

In [ ]:
# Trying to initialize the model with the real name will throw a Pydantic error
try:
    User(name="APTG")
except ValidationError as err:
    pprint(err.errors())

Looks like validation alias and alias are semantically similar!

In [ ]:
%reset -f
from pydantic import *
from pprint import pprint

In [ ]:
class User(BaseModel):
    name: str = Field(validation_alias="username")

In [ ]:
# Use the alias at initialization
# PyLance is giving a false positive!
user = User(username="APTG")  # type: ignore
user

In [ ]:
# Serialization will use the real name
user.model_dump()

In [ ]:
# Field access is also via the real name
user.name

In [ ]:
# Trying to access the alias will throw a plain old Python error
try:
    user.username
except AttributeError as err:
    print(err)

In [ ]:
# Trying to initialize the model with the real name will throw a Pydantic error
try:
    User(name="APTG")
except ValidationError as err:
    pprint(err.errors())

The difference is that I can have multiple validation aliases.

In [ ]:
%reset -f
from pydantic import *
from pprint import pprint

In [ ]:
class User(BaseModel):
    # I am going to include the real name in the alias for consistency
    name: str = Field(validation_alias=AliasChoices("name", "username", "login"))

In [ ]:
# Now I can instantiate the model with the aliased real name
user = User(name="APTG")
user

In [ ]:
user.name

In [ ]:
try:
    user.username
except AttributeError as err:
    print(err)

In [ ]:
User(username="Avilay")  # type: ignore

Now lets try out serialization alias. By default serialization will **not** use the serialization alias! I have to specify a `by_alias` flag during serialization for this to happen. This flag can be set on a per call basis everytime I serialize, or it can be set in the model config.

In [ ]:
class Song(BaseModel):
    name: str = Field(serialization_alias="title")

    # model_config = ConfigDict(serialize_by_alias=True)

In [ ]:
song = Song(name="Abracadabra")
song

In [ ]:
song.model_dump()

In [ ]:
song.model_dump(by_alias=True)